# Khmer History RAG - Part 1: Preprocessing + Splitting

## Import

In [1]:
import sys
!{sys.executable} -m pip install khmer-nltk

In [2]:
import re
import json
import unicodedata
from collections import Counter
 
import wikipediaapi
from khmernltk import sentence_tokenize, word_tokenize

In [3]:
USER_AGENT = "KhmerHistoryRAGProject/1.0 (2024384ly@aupp.edu.kh; ITM454 Project)"
PAGE_TITLES = ["ប្រវត្តិសាស្ត្រកម្ពុជា"]   # add more pages here later
 
# Sections with no real historical content (check against the tree printed in CELL 3;
# edit this list if your page uses different heading names)
SKIP_HEADINGS = {
    "ឯកសារយោង",            # References
    "កំណត់ចំណាំ",          # Notes
    "តំណភ្ជាប់ខាងក្រៅ",     # External links
    "សូមមើលផងដែរ",         # See also
    "គន្ថនិទ្ទេស",          # Bibliography
}

## Fetching API

In [4]:
def clean_heading(title: str) -> str:
    """Remove wiki markup such as '**', '==' from headings."""
    title = re.sub(r"[*=#]+", "", title)
    return re.sub(r"\s+", " ", title).strip()
 
 
def walk_sections(sections, parent_path=()):
    """Recursively yield (path_tuple, text). section.text excludes sub-sections,
    so we must descend into section.sections ourselves."""
    for s in sections:
        title = clean_heading(s.title)
        path = parent_path + (title,)
        if title in SKIP_HEADINGS:
            continue
        yield path, s.text
        yield from walk_sections(s.sections, path)
 
 
def fetch_pages(page_titles=PAGE_TITLES):
    wiki = wikipediaapi.Wikipedia(user_agent=USER_AGENT, language="km")
    records = []
    for title in page_titles:
        page = wiki.page(title)
        if not page.exists():
            print(f"[WARN] page not found: {title}")
            continue
        print(f"[OK] {page.title}")
        # the intro (text before the first heading) is kept as its own record
        if page.summary.strip():
            records.append({"page_title": page.title, "url": page.fullurl,
                            "section_path": ("សេចក្ដីផ្ដើម",), "text": page.summary})
        for path, text in walk_sections(page.sections):
            records.append({"page_title": page.title, "url": page.fullurl,
                            "section_path": path, "text": text})
    return records
 
 

## Cleaning & Preprocessing

In [7]:
ZERO_WIDTH = re.compile(r"[\u200b\u200c\u200d\u2060\ufeff]")
CITATION = re.compile(r"\[(?:\d+|[ក-អ]+|សំណើ|ត្រូវការការអះអាង)\]")   # [1] [ក] ...
HTML_TAG = re.compile(r"<[^>]+>")
TEMPLATE = re.compile(r"\{\{[^}]*\}\}")
HEADING_MARKUP = re.compile(r"^\s*=+.*?=+\s*$", re.MULTILINE)
 
 
def clean_and_normalize_khmer(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = ZERO_WIDTH.sub("", text)
    text = HEADING_MARKUP.sub(" ", text)       # leftover '== Title =='
    text = TEMPLATE.sub(" ", text)
    text = HTML_TAG.sub(" ", text)
    text = CITATION.sub("", text)
    text = text.replace("**", "").replace("''", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()
 

## Splitting into chunks

In [9]:
word_cache = {}
 
def count_words(text: str) -> int:
    """Real Khmer word count using khmernltk (Khmer has no spaces between words)."""
    if text not in word_cache:
        word_cache[text] = sum(1 for w in word_tokenize(text) if w.strip())
    return word_cache[text]
 
 
def split_long_sentence(sentence: str, max_words: int):
    """Safety net: if the sentence splitter fails and returns one huge 'sentence',
    cut it on word boundaries so no piece exceeds max_words."""
    if count_words(sentence) <= max_words:
        return [sentence]
    words = [w for w in word_tokenize(sentence) if w.strip()]
    return ["".join(words[i:i + max_words]) for i in range(0, len(words), max_words)]
 
 
def split_section(text, chunk_size=180, overlap_words=30, min_words=40):
    """Sentence-aware sliding window.
    chunk_size    : target max words per chunk
    overlap_words : words repeated from the end of the previous chunk
    min_words     : a tiny last chunk gets merged into the previous one
    Returns a list of dicts: {"sentences": [...], "n_overlap": int}
    """
    sentences = []
    for s in sentence_tokenize(text):
        s = s.strip()
        if s:
            sentences.extend(split_long_sentence(s, chunk_size))
 
    chunks, current, current_words, n_overlap = [], [], 0, 0
    for sent in sentences:
        w = count_words(sent)
        if current and current_words + w > chunk_size:
            chunks.append({"sentences": current, "n_overlap": n_overlap})
            # build overlap from the tail of the finished chunk
            tail, tail_words = [], 0
            for s in reversed(current):
                sw = count_words(s)
                if tail and tail_words + sw > overlap_words:
                    break
                tail.insert(0, s)
                tail_words += sw
            current, current_words, n_overlap = list(tail), tail_words, len(tail)
        current.append(sent)
        current_words += w
 
    if current and len(current) > n_overlap:          # has new (non-overlap) content
        last = {"sentences": current, "n_overlap": n_overlap}
        new_words = sum(count_words(s) for s in current[n_overlap:])
        if chunks and new_words < min_words:           # merge tiny tail into previous
            chunks[-1]["sentences"].extend(current[n_overlap:])
        else:
            chunks.append(last)
    return chunks
 
 
def process_and_split_pipeline(records, chunk_size=180, overlap_words=30, min_words=40):
    final = []
    for rec_idx, rec in enumerate(records):
        text = clean_and_normalize_khmer(rec["text"])
        if len(text) < 20:
            continue
        heading = " > ".join(rec["section_path"])
        for chunk_idx, ch in enumerate(split_section(text, chunk_size, overlap_words, min_words)):
            payload = " ".join(ch["sentences"])
            final.append({
                "chunk_id": f"km-hist-{rec_idx:03d}-{chunk_idx:02d}",
                "text_payload": payload,
                # heading + text is what teammate 2 should embed (gives era context)
                "text_for_embedding": f"{heading}: {payload}",
                "metadata": {
                    "source_url": rec["url"],
                    "document_domain": "Khmer Wikipedia API",
                    "page_title": rec["page_title"],
                    "historical_era_heading": heading,
                    "section_index": rec_idx,
                    "chunk_index": chunk_idx,
                    "n_words": count_words(payload),
                    "n_chars": len(payload),
                },
            })
    return final
 
 

## Verify and Export

In [10]:
if __name__ == "__main__":
    records = fetch_pages()
    print(f"\nRecords fetched: {len(records)}")
    for r in records:
        print("  ", " > ".join(r["section_path"]), f"({len(r['text'])} chars)")
 
    chunks = process_and_split_pipeline(records)
 
    words = [c["metadata"]["n_words"] for c in chunks]
    print(f"\nTotal chunks: {len(chunks)}")
    print(f"Words per chunk  min/avg/max: {min(words)} / {sum(words)//len(words)} / {max(words)}")
    print("\nChunks per section:")
    for h, n in Counter(c["metadata"]["historical_era_heading"] for c in chunks).items():
        print(f"  {n:>3}  {h}")
 
    print("\nSample chunk:")
    print(json.dumps(chunks[min(2, len(chunks) - 1)], ensure_ascii=False, indent=2)[:900])
 
    with open("khmer_history_chunks.json", "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)
    print("\nSaved -> khmer_history_chunks.json")

[OK] ប្រវត្តិសាស្ត្រកម្ពុជា

Records fetched: 15
   សេចក្ដីផ្ដើម (5943 chars)
   បុរេប្រវត្តិសាស្ត្រ និង ប្រវត្តិសាស្ត្រដើមដំបូង (2444 chars)
   ហ៊្វូណន រឺ ភ្វូណាម (9744 chars)
   កម្វុជ (1238 chars)
   កម្វុជទេឝ (3452 chars)
   យុគអន្ធកាលនៃកម្ពុជា (2196 chars)
   សម័យអាណានិគមនិយមបារាំង (1016 chars)
   រាជការលើកទីមួយរបស់ព្រះសីហនុ១៩៥៣-១៩៧០ (4849 chars)
   សាធារណរដ្ឋខ្មែរនិងសង្គ្រាម (7793 chars)
   កម្ពុជាប្រជាធិបតេយ្យ (១៩៧៥-១៩៧៩) (5024 chars)
   កម្ពុជាប្រជាធិបតេយ្យ (១៩៧៥-១៩៧៩) > ការជាប់ទាក់ទងខាងសង្គម និង វប្បធម៌នៃរបបនេះ (2699 chars)
   សាធារណរដ្ឋប្រជាមានិតកម្ពុជា (937 chars)
   កម្ពុជាសម័យទំនើប:១៩៩៣-បច្ចុប្បន្ន (2889 chars)
   មើលផងដែរ (193 chars)
   តំណភ្ជាប់ក្រៅ (68 chars)


| 2026-09-30 15:17:06,233 | INFO | khmer-nltk | Loaded model from /Users/soryapheakly/miniconda3/envs/itm454/lib/python3.12/site-packages/khmernltk/word_tokenize/sklearn_crf_ner_10000.sav |



Total chunks: 78
Words per chunk  min/avg/max: 9 / 158 / 196

Chunks per section:
    9  សេចក្ដីផ្ដើម
    4  បុរេប្រវត្តិសាស្ត្រ និង ប្រវត្តិសាស្ត្រដើមដំបូង
   17  ហ៊្វូណន រឺ ភ្វូណាម
    2  កម្វុជ
    5  កម្វុជទេឝ
    3  យុគអន្ធកាលនៃកម្ពុជា
    1  សម័យអាណានិគមនិយមបារាំង
    7  រាជការលើកទីមួយរបស់ព្រះសីហនុ១៩៥៣-១៩៧០
   12  សាធារណរដ្ឋខ្មែរនិងសង្គ្រាម
    7  កម្ពុជាប្រជាធិបតេយ្យ (១៩៧៥-១៩៧៩)
    4  កម្ពុជាប្រជាធិបតេយ្យ (១៩៧៥-១៩៧៩) > ការជាប់ទាក់ទងខាងសង្គម និង វប្បធម៌នៃរបបនេះ
    1  សាធារណរដ្ឋប្រជាមានិតកម្ពុជា
    4  កម្ពុជាសម័យទំនើប:១៩៩៣-បច្ចុប្បន្ន
    1  មើលផងដែរ
    1  តំណភ្ជាប់ក្រៅ

Sample chunk:
{
  "chunk_id": "km-hist-000-02",
  "text_payload": "ក៏ត្រូវបានគេសាងសង់ឡើង សំណង់ដែលគួរឱ្យកត់សំគាល់បំផុតគឺប្រាសាទអង្គរវត្ត។ អធិរាជាណាចក្រខ្មែរត្រូវបានបង្កើតឡើងនៅកៀកនឹងដើមសតវត្សទី៩។ ខ្លឹមសារយោងយកតាមត្រង់សេចក្ដីនិយាយអំពីពិធីអាទ្យាចរិយកម្ម និង ប្រសិទ្ធីបែបទេវៈជាពិធីធ្វើឡើងដើម្បីបញ្ជាក់ភាពស្របច្បាប់ខាងនយោបាយដល់ស្ថាបនិកនៃអាណាចក្រនេះ ព្រះបាទជយវម៌្មទី៣ ដែលធ្វើនៅលើកំពូលភ្នំគូលែន (មហេន្ទ្របវ៌ត) នៅឆ្នាំគ.ស